# 05 — RAG multilingue avec citations

Le retrieval hybride fusionne similarité dense et score BM25. La réponse doit reconnaître explicitement lorsque les documents ne suffisent pas.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA = PROJECT_ROOT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
INDEX = DATA / "index"
for directory in (RAW, PROCESSED, INDEX):
    directory.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

In [ ]:
import json
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

chunks = json.loads((INDEX / "chunks.json").read_text(encoding="utf-8"))
embeddings = np.load(INDEX / "embeddings.npy")
with (INDEX / "bm25.pkl").open("rb") as stream:
    bm25 = pickle.load(stream)
encoder = SentenceTransformer("intfloat/multilingual-e5-base")

def minmax(values: np.ndarray) -> np.ndarray:
    if len(values) == 0 or float(values.max() - values.min()) == 0:
        return np.zeros_like(values, dtype=float)
    return (values - values.min()) / (values.max() - values.min())

def retrieve(question: str, k: int = 5, alpha: float = 0.65) -> list[dict]:
    query_vector = encoder.encode([f"query: {question}"], normalize_embeddings=True)[0]
    dense = embeddings @ query_vector
    lexical = np.asarray(bm25.get_scores(question.lower().split()))
    scores = alpha * minmax(dense) + (1 - alpha) * minmax(lexical)
    best = np.argsort(scores)[::-1][:k]
    return [{**chunks[i], "score": float(scores[i])} for i in best]

question = "Quels documents répondent à ma question juridique ?"
contexts = retrieve(question) if chunks else []
[(c["file"], c["page"], round(c["score"], 3)) for c in contexts]

In [ ]:
def build_grounded_prompt(question: str, contexts: list[dict], language: str = "français") -> str:
    evidence = "\n\n".join(
        f"[Source {i} — {c['file']}, page {c['page']}]\n{c['text_normalized']}"
        for i, c in enumerate(contexts, start=1)
    )
    instructions = [
        "Tu es un assistant de recherche documentaire en droit marocain.",
        f"Réponds en {language}. Utilise uniquement les extraits fournis.",
        "Cite chaque affirmation sous la forme [Source N].",
        "Si les sources sont insuffisantes ou contradictoires, dis-le clairement.",
        "Ne présente jamais la réponse comme un avis juridique professionnel.",
        f"Question : {question}",
        f"Extraits :\n{evidence}",
    ]
    return "\n\n".join(instructions)

prompt = build_grounded_prompt(question, contexts)
print(prompt[:3000])

La cellule suivante du projet pourra brancher un LLM local ou une API. Le prompt, le retrieval et les citations restent indépendants du fournisseur afin de faciliter les comparaisons.